# SaaS Churn & Revenue Analysis
## Project Overview
This project analyzes subscription, customer, revenue, and payment data from a fictional SaaS business.
The goal is to identify patterns in customer cancellations, recurring revenue, and payment performance using Python and Pandas.

## Data Preparation
The analysis uses a SQLite database containing subscription, customer account, plan, invoice, and payment data.
The subscription data is loaded into Pandas for analysis and date fields are converted into datetime format where required.

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect(
    r"C:\Users\asgaj\OneDrive\Desktop\saas-churn-analytics\data\saas-subscriptions.sqlite"
)

print("Database connected successfully!")

Database connected successfully!


In [2]:
subscriptions = pd.read_sql_query(
    "SELECT * FROM subscriptions",
    conn
)

subscriptions.head()

,subscription_id,account_id,plan_id,started_at,ended_at,status,seats,mrr
0,S000001,A000001,PLN_GROWTH,2024-06-11,,active,13,199
1,S000002,A000002,PLN_STARTER,2025-04-07,,active,3,49
2,S000003,A000003,PLN_GROWTH,2024-07-24,,active,7,199
3,S000004,A000004,PLN_STARTER,2024-01-14,,active,4,49
4,S000005,A000005,PLN_SCALE,2025-09-30,,active,79,799


In [3]:
subscriptions.shape

(5000, 8)

In [4]:
subscriptions.isnull().sum()

subscription_id    0
account_id         0
plan_id            0
started_at         0
ended_at           0
status             0
seats              0
mrr                0
dtype: int64

In [5]:
subscriptions.dtypes

subscription_id      str
account_id           str
plan_id              str
started_at           str
ended_at             str
status               str
seats              int64
mrr                int64
dtype: object

In [6]:
subscriptions["started_at"] = pd.to_datetime(subscriptions["started_at"])
subscriptions["ended_at"] = pd.to_datetime(subscriptions["ended_at"])

subscriptions.dtypes

subscription_id               str
account_id                    str
plan_id                       str
started_at         datetime64[us]
ended_at           datetime64[us]
status                        str
seats                       int64
mrr                         int64
dtype: object

In [7]:
subscriptions["status"].value_counts()

status
active       4035
cancelled     965
Name: count, dtype: int64

## Churn Analysis
### Overall Cancellation Rate
Calculate the percentage of subscriptions that have been cancelled to understand the overall churn level.

In [8]:
cancellation_rate = (
    subscriptions["status"].eq("cancelled").mean() * 100
)

print(f"Cancellation rate: {cancellation_rate:.2f}%")

Cancellation rate: 19.30%


In [9]:
plan_churn = (
    subscriptions.groupby("plan_id")
    .agg(
        total_subscriptions=("subscription_id", "count"),
        cancelled=("status", lambda x: (x == "cancelled").sum())
    )
)

plan_churn["cancellation_rate"] = (
    plan_churn["cancelled"] / plan_churn["total_subscriptions"] * 100
)

plan_churn = plan_churn.sort_values(
    "cancellation_rate",
    ascending=False
)

plan_churn

,total_subscriptions,cancelled,cancellation_rate
plan_id,,,
PLN_ENTERPRISE,283,60,21.201413
PLN_STARTER,2433,482,19.810933
PLN_SCALE,675,127,18.814815
PLN_GROWTH,1609,296,18.396520


In [10]:
plans = pd.read_sql_query(
    "SELECT plan_id, plan_name FROM plans",
    conn
)

plan_churn = plan_churn.reset_index().merge(
    plans,
    on="plan_id"
)

plan_churn[
    ["plan_name", "total_subscriptions", "cancelled", "cancellation_rate"]
]

,plan_name,total_subscriptions,cancelled,cancellation_rate
0,Enterprise,283,60,21.201413
1,Starter,2433,482,19.810933
2,Scale,675,127,18.814815
3,Growth,1609,296,18.396520


In [11]:
segment_churn = (
    subscriptions
    .merge(
        pd.read_sql_query(
            "SELECT account_id, segment FROM accounts",
            conn
        ),
        on="account_id"
    )
    .groupby("segment")
    .agg(
        total_subscriptions=("subscription_id", "count"),
        cancelled=("status", lambda x: (x == "cancelled").sum())
    )
)

segment_churn["cancellation_rate"] = (
    segment_churn["cancelled"]
    / segment_churn["total_subscriptions"]
    * 100
)

segment_churn = segment_churn.sort_values(
    "cancellation_rate",
    ascending=False
)

segment_churn

,total_subscriptions,cancelled,cancellation_rate
segment,,,
SMB,3522,699,19.846678
enterprise,305,59,19.344262
mid_market,1173,207,17.647059


In [12]:
channel_churn = (
    subscriptions
    .merge(
        pd.read_sql_query(
            "SELECT account_id, acquisition_channel FROM accounts",
            conn
        ),
        on="account_id"
    )
    .groupby("acquisition_channel")
    .agg(
        total_subscriptions=("subscription_id", "count"),
        cancelled=("status", lambda x: (x == "cancelled").sum())
    )
)

channel_churn["cancellation_rate"] = (
    channel_churn["cancelled"]
    / channel_churn["total_subscriptions"]
    * 100
)

channel_churn = channel_churn.sort_values(
    "cancellation_rate",
    ascending=False
)

channel_churn

,total_subscriptions,cancelled,cancellation_rate
acquisition_channel,,,
outbound,1008,217,21.527778
organic,944,201,21.292373
referral,1040,194,18.653846
paid_search,972,179,18.415638
partner,1036,174,16.795367


## Revenue Analysis
### Monthly Recurring Revenue (MRR) by Plan
Analyze active subscriptions to compare total and average MRR across different SaaS plans.

In [13]:
active_mrr = (
    subscriptions[subscriptions["status"] == "active"]
    .merge(
        plans,
        on="plan_id"
    )
    .groupby("plan_name")
    .agg(
        subscriptions=("subscription_id", "count"),
        total_mrr=("mrr", "sum"),
        avg_mrr=("mrr", "mean")
    )
)

active_mrr["total_mrr"] = active_mrr["total_mrr"].round(2)
active_mrr["avg_mrr"] = active_mrr["avg_mrr"].round(2)

active_mrr = active_mrr.sort_values(
    "total_mrr",
    ascending=False
)

active_mrr

,subscriptions,total_mrr,avg_mrr
plan_name,,,
Enterprise,223,558069,2502.55
Scale,548,458912,837.43
Growth,1313,272071,207.21
Starter,1951,98389,50.43


### Revenue Contribution by Plan
Compare active subscription volume with total MRR to identify which plans contribute the most recurring revenue.

In [14]:
revenue_share = active_mrr.copy()

revenue_share["revenue_share"] = (
    revenue_share["total_mrr"]
    / revenue_share["total_mrr"].sum()
    * 100
)

revenue_share["revenue_share"] = revenue_share["revenue_share"].round(2)

revenue_share[["subscriptions", "total_mrr", "revenue_share"]]

,subscriptions,total_mrr,revenue_share
plan_name,,,
Enterprise,223,558069,40.22
Scale,548,458912,33.08
Growth,1313,272071,19.61
Starter,1951,98389,7.09


## Payment Analysis
### Payment Success vs Failure
Analyze payment attempts to understand the overall payment success rate and identify potential payment-related revenue risk.

In [15]:
payment_data = pd.read_sql_query(
    """
    SELECT
        payment_status,
        COUNT(*) AS payment_attempts
    FROM payments
    GROUP BY payment_status
    """,
    conn
)

payment_data["percentage"] = (
    payment_data["payment_attempts"]
    / payment_data["payment_attempts"].sum()
    * 100
)

payment_data["percentage"] = payment_data["percentage"].round(2)

payment_data

,payment_status,payment_attempts,percentage
0,failed,7186,8.23
1,succeeded,80101,91.77


## Key Findings
The analysis highlights several important patterns in the SaaS business:
- The overall subscription cancellation rate is **19.30%**.
- **Enterprise** has the highest cancellation rate among plans at **21.20%**.
- **SMB** customers have the highest cancellation rate among segments at **19.85%**.
- **Outbound** acquisition has the highest cancellation rate at **21.53%**, while Partner has the lowest at **16.80%**.
- Enterprise generates the highest total MRR and contributes **40.22%** of active MRR.
- Enterprise and Scale together contribute **73.30%** of active MRR despite having far fewer subscriptions than Starter.
- **91.77%** of payment attempts succeed, while **8.23%** fail.